# Practica 28: Lectura y escritura de archivos CSV, Excel y JSON
#### Objetivo: Practicar cómo leer información, guardarla en listas, consultar las primeras y últimas 10 filas y crear nuevos archivos con algunas columnas seleccionadas.
#### Trabajaremos con los archivos proporcionados para FIFA, los libros más vendidos de Amazon y los subtítulos de Game of Thrones.
###### Nota: En la consigna aparecen topBestSellingNovels.xlsx y GOT Subtitles.json, pero los archivos entregados tienen otros nombres. Por eso utilizaré Amazon_top100_bestselling_books_2009to2021.xlsx y los siete archivos GOT Subtitles_season1.json a GOT Subtitles_season7.json.

## 1. Preparación: importar librerías y cambiar de directorio
#### Utilizaré os para cambiar de carpeta, csv para los archivos CSV, pandas para Excel y json para los subtítulos.

In [ ]:
import os
import csv
import json
import pandas as pd

# Cambio el directorio a mi carpeta Downloads
from pathlib import Path
# Run Jupyter from the repository root; place course inputs here.
data_dir = Path.cwd() / "data-analysis" / "datasets" / "module-28"
data_dir.mkdir(parents=True, exist_ok=True)
os.chdir(data_dir)

print("Directorio de trabajo:", os.getcwd())

### Comprobar que los archivos estén en Downloads
#### Antes de comenzar, reviso que los nombres sean correctos. Si aparece Falta, debo mover ese archivo a la carpeta Downloads

In [ ]:
archivos = [
    "fifa_eda.csv",
    "Amazon_top100_bestselling_books_2009to2021.xlsx"
]

for temporada in range(1, 8):
    archivos.append(f"GOT Subtitles_season{temporada}.json")

for nombre in archivos:
    if os.path.exists(nombre):
        print("Encontrado:", nombre)
    else:
        print("Falta:", nombre)

## 2. Archivo CSV: fifa_eda.csv
### 2.1 Leer el CSV y guardarlo en una lista
#### csv.DictReader() lee cada fila como un diccionario, utilizando los nombres de las columnas como claves. list() guarda todas las filas dentro de una lista.

In [ ]:
with open("fifa_eda.csv", "r", encoding="utf-8-sig", newline="") as archivo:
    lector = csv.DictReader(archivo)
    lista_fifa = list(lector)

print("Total de registros FIFA:", len(lista_fifa))
print("Columnas:", list(lista_fifa[0].keys()))

### 2.2 Mostrar las primeras 10 filas

In [ ]:
for fila in lista_fifa[:10]:
    print(fila)

### 2.3 Mostrar las últimas 10 filas

In [ ]:
for fila in lista_fifa[-10:]:
    print(fila)

### 2.4 Seleccionar cuatro columnas y crear otro CSV
#### Selecciono Name, Nationality, Overall y Club. Después guardo los datos con csv.DictWriter().

In [ ]:
columnas_fifa = ["Name", "Nationality", "Overall", "Club"]
lista_fifa_nueva = []

for fila in lista_fifa:
    nueva_fila = {
        "Name": fila["Name"],
        "Nationality": fila["Nationality"],
        "Overall": fila["Overall"],
        "Club": fila["Club"]
    }
    lista_fifa_nueva.append(nueva_fila)

with open("fifa_columnas_seleccionadas.csv", "w", encoding="utf-8", newline="") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=columnas_fifa)
    escritor.writeheader()
    escritor.writerows(lista_fifa_nueva)

print("Archivo creado: fifa_columnas_seleccionadas.csv")
print("Ejemplo:", lista_fifa_nueva[0])

## 3. Archivo Excel: Amazon_top100_bestselling_books_2009to2021.xlsx
### 3.1 Leer el Excel y guardar sus registros en una lista
#### Utilizo pd.read_excel() para leer el archivo y to_dict(orient="records") para convertirlo en una lista de diccionarios. El archivo tiene una primera columna sin nombre que representa un índice; para el archivo nuevo escogeré únicamente columnas con información de los libros.

In [ ]:
tabla_libros = pd.read_excel("Amazon_top100_bestselling_books_2009to2021.xlsx")
lista_libros = tabla_libros.to_dict(orient="records")

print("Total de registros de libros:", len(lista_libros))
print("Columnas:", tabla_libros.columns.tolist())

### 3.2 Mostrar las primeras 10 filas

In [ ]:
for fila in lista_libros[:10]:
    print(fila)

### 3.3 Mostrar las últimas 10 filas

In [ ]:
for fila in lista_libros[-10:]:
    print(fila)

### 3.4 Seleccionar cuatro columnas y generar un Excel nuevo
#### Me quedo con el título, autor, precio y año; después guardo el resultado con to_excel().

In [ ]:
columnas_libros = ["title", "author", "price", "year"]

# Selecciono solamente las cuatro columnas
libros_seleccionados = tabla_libros[columnas_libros]

# También puedo guardar esas filas seleccionadas como una lista
lista_libros_nueva = libros_seleccionados.to_dict(orient="records")

# Creo un archivo Excel nuevo sin incluir el índice
libros_seleccionados.to_excel("libros_columnas_seleccionadas.xlsx", index=False)

print("Archivo creado: libros_columnas_seleccionadas.xlsx")
print("Ejemplo:", lista_libros_nueva[0])

## 4. Archivos JSON: subtítulos de Game of Thrones
#### Los datos están divididos en siete archivos, uno por temporada. Cada archivo contiene diccionarios anidados: el nombre del episodio es la clave principal y dentro de cada episodio aparecen números de línea asociados con sus textos.
#### Para poder trabajar con filas y columnas, convertiré esa estructura en una lista de diccionarios. Cada fila tendrá temporada, episodio, número de línea y texto.
### 4.1 Leer los siete archivos JSON y guardar los subtítulos en una lista

In [ ]:
lista_subtitulos = []

for temporada in range(1, 8):
    nombre_archivo = f"GOT Subtitles_season{temporada}.json"

    with open(nombre_archivo, "r", encoding="utf-8") as archivo:
        datos_temporada = json.load(archivo)

    # Recorro los episodios de la temporada
    for episodio in sorted(datos_temporada):
        dialogos = datos_temporada[episodio]

        # Ordeno los números de las líneas para que no aparezcan 1, 10, 100, 2...
        for numero in sorted(dialogos, key=int):
            fila = {
                "Temporada": temporada,
                "Episodio": episodio,
                "Numero": int(numero),
                "Texto": dialogos[numero]
            }
            lista_subtitulos.append(fila)

print("Total de líneas de subtítulos:", len(lista_subtitulos))
print("Primer registro:", lista_subtitulos[0])

### 4.2 Mostrar las primeras 10 filas

In [ ]:
for fila in lista_subtitulos[:10]:
    print(fila)

### 4.3 Mostrar las últimas 10 filas

In [ ]:
for fila in lista_subtitulos[-10:]:
    print(fila)

### 4.4 Seleccionar tres campos y crear un JSON nuevo
#### Conservo solamente los campos Temporada, Episodio y Texto de las cuatro columnas de la lista original. Así practico cómo seleccionar información en un JSON. Después utilizo json.dump() para generar el nuevo archivo.

In [ ]:
columnas_json = ["Temporada", "Episodio", "Texto"]
lista_json_nueva = []

for fila in lista_subtitulos:
    nueva_fila = {
        "Temporada": fila["Temporada"],
        "Episodio": fila["Episodio"],
        "Texto": fila["Texto"]
    }
    lista_json_nueva.append(nueva_fila)

with open("GOT_subtitulos_seleccionados.json", "w", encoding="utf-8") as archivo:
    json.dump(lista_json_nueva, archivo, ensure_ascii=False, indent=2)

print("Archivo creado: GOT_subtitulos_seleccionados.json")
print("Ejemplo:", lista_json_nueva[0])

## 5. Comprobar los archivos generados
#### Al terminar, verifico que los tres archivos nuevos se hayan guardado en Downloads.

In [ ]:
archivos_nuevos = [
    "fifa_columnas_seleccionadas.csv",
    "libros_columnas_seleccionadas.xlsx",
    "GOT_subtitulos_seleccionados.json"
]

for nombre in archivos_nuevos:
    if os.path.exists(nombre):
        print("Creado correctamente:", nombre)
    else:
        print("No encontrado:", nombre)